# When Foundation Models Fall Short: From Off-the-Shelf AI to Problem-Specific Adaptation

<table width="100%" bgcolor="#4A148C"><tr><td>
<font color="white" size="6"><b>When Foundation Models Fall Short</b></font><br>
<font color="white" size="4">From Off-the-Shelf AI to Problem-Specific Adaptation</font><br>
<font color="white">AMLD 2027 · EPFL, 11 March 2027 &nbsp;·&nbsp; SDS 2027 · Zurich, 1 June 2027</font>
<br><br>
<font color="white"><b>The question:</b> what can modern AI solve out of the box, when do its limitations
start to matter, and what should we do next?</font>
<br><br>
<font color="white"><b>At the end of this notebook you will be able to:</b></font>
<ol>
<li><font color="white">Judge when a vision foundation model is good enough out of the box, and recognise
the signs that it is not.</font></li>
<li><font color="white">Adapt it with cheap, incomplete supervision (scribbles, missing labels,
pseudo-labels, explanations, prompts), and explain what each approach uses and assumes.</font></li>
<li><font color="white">Compare approaches on accuracy, annotation effort, reliability and compute, and
decide what to do next on a new problem, including your own images.</font></li>
</ol>
<font color="white">Levels: 🟢 beginner · 🟡 intermediate (the default path) · 🔴 advanced, optional.
&nbsp; ✍️ = you write a few lines &nbsp; 🖍️ = you draw &nbsp; 👥 = compare with your neighbour.</font>
<br><br>
<font color="white"><b>Run <code>setup_check.ipynb</code> first.</b> It was shared with you the week
before: it puts the models, images and precomputed files in your Google Drive, so nothing is downloaded
in the room. If you have not run it, do it now (about 5 minutes), then come back here.</font>
</td></tr></table>

**Objective:** load the cached models and data; see the question and outcomes.

**What you need:** basic ML (training, validation, objectives, evaluation), some Python, this notebook open
in Google Colab with **Runtime → Change runtime type → T4 GPU**. No computer-vision background is needed.
Every step also runs on a CPU, just slower, and each training cell offers a precomputed checkpoint.

**The plan.** Block 1 (sections 1–3): a foundation model segments pets almost perfectly, then meets a
specialised task and falls short. Block 2 (sections 4–10): the ways out, from scribbles to prompting the
model directly, all compared on the same hidden ground truth, then on your own images.

<table width="100%" bgcolor="#FFF8E1"><tr><td><font color="#5D4037">🧾 <b>Recipe and sources.</b>
The feature extractor is <b>DINOv3</b> ViT-S/16 (Meta, 2025, via <code>timm</code>) and the promptable
segmenter is <b>SAM 2.1</b> hiera-small (Meta, Apache-2.0). The scribble idea follows the MIT
"DINOv2 + scribbles" recipe: a tiny head on frozen patch features, trained with a loss that ignores
unlabelled pixels. Datasets and licences are listed in section 10 and in <code>data/ATTRIBUTION.md</code>.
</font></td></tr></table>

## 0. Setup & the question

The next cell detects where you are (Colab or not, GPU or not), fixes the random seed, creates the
`RESULTS` dictionary every section writes into, and defines `use()`, the helper that lets you write your
own code without ever blocking the notebook. Run it once.

In [ ]:
# @title Setup (run me first) { display-mode: "form" }
import ast
import inspect
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NB_SMOKE") == "1"  # automated test: tiny data, CPU, seconds. Never set this by hand.
SEED = 42

# Where the cache lives. setup_check.ipynb filled MyDrive/fm-workshop-cache/ in Colab; the smoke run uses the
# small fixtures committed in data/smoke/; a local checkout uses data/cache/ (scripts/prepare_data.py).
if IN_COLAB and not SMOKE:
    CACHE_DIR = Path("/content/drive/MyDrive/fm-workshop-cache")
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception as err:  # noqa: BLE001 - one plain line, then a plain message below
        print(f"(Drive mount error: {type(err).__name__}: {err})")
    if not CACHE_DIR.is_dir():
        print(
            "⚠️ Google Drive is not mounted or the cache folder is missing. Run this cell again and click "
            "'Allow' in the pop-up. If the folder is still missing, run setup_check.ipynb first (5 minutes)."
        )
    if (CACHE_DIR / "hf").is_dir():
        os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "hf")  # model weights, before torch/timm are imported
        os.environ["HF_HUB_OFFLINE"] = "1"  # nothing is downloaded in the room (N-030)
    DATA_DIR, PRECOMPUTED = CACHE_DIR / "data", CACHE_DIR / "precomputed"
    SPECIALISED_JSON = CACHE_DIR / "specialised.json"
else:
    CACHE_DIR = Path("data/smoke" if SMOKE else "data/cache")
    DATA_DIR = CACHE_DIR if SMOKE else CACHE_DIR / "workshop"
    PRECOMPUTED = CACHE_DIR / "precomputed"
    SPECIALISED_JSON = Path("data/specialised.json")

if IN_COLAB:
    try:
        import pytorch_grad_cam  # noqa: F401 - the only package Colab lacks (SPEC §5); installed if missing
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q grad-cam")  # noqa: F821 - %pip, inside a cell

import torch

if SMOKE:
    DEVICE = "cpu"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
    print(
        "ℹ️ No GPU in this session, so everything runs on the CPU: fine, but slower. For a GPU, use "
        "Runtime → Change runtime type → T4 GPU, then run this cell again. Every training cell also "
        "offers our precomputed checkpoint."
    )


def seed_everything(seed: int = SEED) -> None:
    """Fix every random source, so re-running a section from its first cell gives the same result (N-013)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

RESULTS: dict = {"use_fallbacks": {}}  # every section writes its named numbers here (SPEC §9)


def _is_stub(fn) -> bool:
    """True when the function body is still the `...` placeholder (optionally after a docstring)."""
    try:
        tree = ast.parse(inspect.getsource(fn).strip() or "pass")
    except (OSError, TypeError, SyntaxError):
        return False
    node = tree.body[0]
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return False
    def const(n, kind):
        is_const = isinstance(n, ast.Expr) and isinstance(n.value, ast.Constant)
        return is_const and isinstance(n.value.value, kind)

    body = [n for n in node.body if not const(n, str)]  # drop the docstring
    return all(const(n, type(...)) for n in body)


def _same(a, b) -> bool:
    """Compare two results of any common type (numbers, arrays, tensors, dicts, sequences)."""
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        a = a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else a
        b = b.detach().cpu().numpy() if isinstance(b, torch.Tensor) else b
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b, strict=True))
    try:
        return bool(np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=1e-5))
    except (TypeError, ValueError):
        return a == b


def use(fn, reference, check=None, name: str | None = None):
    """Return the function the notebook continues with: yours if it matches the reference, else the reference.

    `check(f)` runs `f` on a small play example and returns what to compare (any numbers, arrays or dicts).
    Nothing here is mandatory: an untouched stub (`...` body) silently selects the reference (N-029).
    """
    name = name or fn.__name__
    if _is_stub(fn):
        RESULTS["use_fallbacks"][name] = True
        print(f"ℹ️ {name}: using the reference solution. Fill the stub above to use your own version.")
        return reference
    if check is None:
        RESULTS["use_fallbacks"][name] = False
        print(f"✅ {name}: using your version (no automatic check for this one).")
        return fn
    try:
        yours, theirs = check(fn), check(reference)
    except Exception as err:  # noqa: BLE001 - the participant's code may fail in any way; keep going
        RESULTS["use_fallbacks"][name] = True
        print(f"❌ {name}: your version raised {type(err).__name__}: {err}. Continuing with the reference.")
        return reference
    if _same(yours, theirs):
        RESULTS["use_fallbacks"][name] = False
        print(f"✅ {name}: matches the reference. Continuing with your version.")
        return fn
    RESULTS["use_fallbacks"][name] = True
    print(f"❌ {name}: {yours} vs reference {theirs}. Continuing with the reference.")
    return reference


if not SPECIALISED_JSON.is_file():
    raise SystemExit(f"{SPECIALISED_JSON} is missing. Run setup_check.ipynb first, then run this cell again.")
SPECIALISED: dict = json.loads(SPECIALISED_JSON.read_text())  # the specialised task; never hard-coded
SPECIALISED["dir"] = DATA_DIR / "specialised"
PETS: dict = {"key": "pets", "name": "Oxford-IIIT Pet", "task": "pet vs background"}
PETS |= {"licence": "CC BY-SA 4.0", "dir": DATA_DIR / "pets"}

print(f"Colab: {IN_COLAB} · smoke mode: {SMOKE} · device: {DEVICE} · seed: {SEED}")
print(f"cache: {CACHE_DIR} · specialised task: {SPECIALISED['name']} ({SPECIALISED['task']})")

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: how <code>use()</code>
works.</b> Several cells later ask you to fill a function whose body is <code>...</code>. The line after
it, <code>fn = use(my_fn, reference=_ref_fn, check=...)</code>, runs both versions on a small play
example and prints ✅ <i>matches</i> or ❌ <i>yours vs reference</i>. Either way the notebook continues:
with your version if it matches, with ours if the stub is untouched or wrong. Writing the code yourself is
where the learning happens; skipping it never breaks anything.</font></td></tr></table>

### The theme

This cell sets the colours used in every figure: cool violet means "it works", warm orange means "it
fails, we adapt", green is the hidden ground truth and teal marks scribbles. A colour means the same
thing in every plot, so you can read a figure without its legend.

In [ ]:
# @title Theme and image helpers { display-mode: "form" }
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image

THEME = {
    "works": "#6A1B9A",  # violet: the foundation model does the job
    "fails": "#E65100",  # orange: it falls short / we adapt
    "truth": "#2E7D32",  # green: hidden ground truth, evaluation only
    "scribble": "#00838F",  # teal: strokes a person drew
    "background": "#9E9E9E",  # grey overlay
    "banner": "#4A148C",
}
mpl.rcParams.update(
    {
        "figure.dpi": 100,
        "figure.facecolor": "white",
        "axes.titlesize": 10,
        "axes.titlecolor": "#311B92",
        "axes.prop_cycle": mpl.cycler(color=[THEME[k] for k in ("works", "fails", "truth", "scribble")]),
    }
)


def hex_rgb(colour: str) -> np.ndarray:
    """'#RRGGBB' -> float array in [0, 1]."""
    return np.array([int(colour[i : i + 2], 16) for i in (1, 3, 5)]) / 255


def overlay(image: np.ndarray, mask: np.ndarray, colour: str = THEME["truth"], alpha: float = 0.55):
    """Blend a binary mask in one theme colour over an RGB uint8 image; returns an RGB uint8 image."""
    out = image.astype(float) / 255
    out[mask] = (1 - alpha) * out[mask] + alpha * hex_rgb(colour)
    return (out * 255).astype(np.uint8)


def read_png(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))


def load_item(dataset: dict, item: dict) -> dict:
    """Read one image with its full mask (evaluation only!), scribbles, visible labels and FOV."""
    root = dataset["dir"]
    out = {
        "name": item["name"],
        "split": item["split"],
        "image": np.array(Image.open(root / item["image"]).convert("RGB")),
        "mask": read_png(root / item["mask"]) > 127,
        "scribbles": read_png(root / item["scribbles"]),
        "visible": read_png(root / item["visible"]) > 127,
        "fov": read_png(root / item["fov"]) > 127 if "fov" in item else None,
    }
    out["hidden"] = out["mask"] & ~out["visible"]
    return out


def load_index(dataset: dict) -> dict:
    """Attach the dataset's index.json (facts + one entry per image) and return it."""
    index_file = dataset["dir"] / "index.json"
    if not index_file.is_file():
        raise SystemExit(f"{index_file} is missing. Run setup_check.ipynb first, then run this cell again.")
    dataset["index"] = json.loads(index_file.read_text())
    dataset["items"] = dataset["index"]["items"]
    dataset["fit"] = [it for it in dataset["items"] if it["split"] == "fit"]
    dataset["held"] = [it for it in dataset["items"] if it["split"] == "held"]
    return dataset["index"]

### The data

This cell reads the two datasets from the cache, the simple task (pets) and the specialised one, and
shows one image of each with its ground truth in green. The ground truth is for evaluation only: from
section 3 on, training sees only incomplete labels.

In [ ]:
for ds in (PETS, SPECIALISED):
    idx = load_index(ds)
    print(
        f"{ds['name']}: {len(ds['items'])} images ({len(ds['fit'])} fit, {len(ds['held'])} held-out) · "
        f"{idx['task']} · licence {idx['licence']}"
    )
PRECOMPUTED_HEADS = json.loads((PRECOMPUTED / "heads" / "index.json").read_text())["heads"]
print(f"precomputed: {len(PRECOMPUTED_HEADS)} head checkpoints in {PRECOMPUTED}")

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for k, ds in enumerate((PETS, SPECIALISED)):
    it = load_item(ds, ds["fit"][0])
    axes[2 * k].imshow(it["image"])
    axes[2 * k].set_title(f"{ds['name']}: {it['name']}")
    axes[2 * k + 1].imshow(overlay(it["image"], it["mask"]))
    axes[2 * k + 1].set_title(f"ground truth: {ds['index']['task']}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> Look at the
two images above. Which task do you expect the foundation model to solve out of the box, and which one
will give it trouble? Write one reason, then keep it: section 2 will tell you whether you were right.
</font></td></tr></table>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Answer in your
head or in a note; a worked answer is folded below.</font></td></tr></table>

**Q1.** 🟢 What is different about the specialised task that could matter for a model trained on everyday
photos? Think about what the objects look like (size, shape, contrast), not about the domain name.

<details><summary>One possible answer</summary>

Everyday photos show compact objects with clear boundaries, exactly what pets are. The specialised task
has structures that are thin, branching and low-contrast, and they cover a small fraction of the image.
A model that learnt "objects" from the web has seen few of those, and its patch features (16 px squares)
are coarser than the structures themselves. Whether that matters is what the next two sections measure.

</details>

### The models

This cell loads the two foundation models from the cache and prints where each one comes from and under
which licence, then wraps them in two small helpers: `dino.features(image, long_side)` gives patch
features, `sam.probability(image, points, labels)` gives a foreground map from point prompts.

In [ ]:
# @title Model loaders { display-mode: "form" }
import timm
import torch.nn.functional as F

DINO_ID = "vit_small_patch16_dinov3.lvd1689m"
DINO_SOURCE = "timm/vit_small_patch16_dinov3.lvd1689m"
DINO_LICENCE = "DINOv3 License (Meta), see the notice below"
SAM_ID = "facebook/sam2.1-hiera-small"  # SPEC §5: SAM 2.1 hiera-small, Apache-2.0, no account needed
SAM_LICENCE = "Apache-2.0"


class Dino:
    """Frozen DINOv3 ViT-S/16: an image in, a grid of 384-dim patch features out."""

    def __init__(self, device: str = DEVICE):
        t = time.monotonic()
        self.device = device
        model = timm.create_model(DINO_ID, pretrained=True, num_classes=0, dynamic_img_size=True)
        self.model = model.eval().to(device)
        cfg = timm.data.resolve_data_config({}, model=self.model)
        self.mean = torch.tensor(cfg["mean"]).view(1, 3, 1, 1).to(device)
        self.std = torch.tensor(cfg["std"]).view(1, 3, 1, 1).to(device)
        self.n_prefix = self.model.num_prefix_tokens
        self.dim = self.model.num_features
        dt = time.monotonic() - t
        print(f"DINOv3 ViT-S/16 · source {DINO_SOURCE} · licence {DINO_LICENCE} · {dt:.0f} s")

    @torch.no_grad()
    def features(self, image: np.ndarray, long_side: int) -> torch.Tensor:
        """Patch features [D, h, w] for the image resized so its long side is `long_side` (multiple of 16)."""
        x = torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1).float().unsqueeze(0) / 255
        h, w = image.shape[:2]
        s = long_side / max(h, w)
        H, W = max(16, round(h * s / 16) * 16), max(16, round(w * s / 16) * 16)
        x = F.interpolate(x.to(self.device), size=(H, W), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        f = self.model.forward_features(x)[0, self.n_prefix :]
        return f.reshape(H // 16, W // 16, -1).permute(2, 0, 1).contiguous().float()


class Sam:
    """SAM 2.1 (promptable segmentation): an image plus points or a box in, a foreground probability out."""

    def __init__(self, device: str = DEVICE):
        from transformers import Sam2Model, Sam2Processor

        t = time.monotonic()
        self.device = device
        self.proc = Sam2Processor.from_pretrained(SAM_ID)
        self.model = Sam2Model.from_pretrained(SAM_ID).eval().to(device)
        self._cache: tuple[np.ndarray, torch.Tensor] | None = None  # (the last image, its embeddings)
        print(f"SAM 2.1 hiera-small · source {SAM_ID} · licence {SAM_LICENCE} · {time.monotonic() - t:.0f} s")

    @torch.no_grad()
    def probability(
        self,
        image: np.ndarray,
        points: list[list[float]] | None = None,
        labels: list[int] | None = None,
        box: list[float] | None = None,
    ) -> np.ndarray:
        """Foreground probability (H x W float) for points (x, y; label 1 = fg, 0 = bg) and/or a box."""
        kw = {}
        if points:
            kw["input_points"], kw["input_labels"] = [[points]], [[labels]]
        if box is not None:
            kw["input_boxes"] = [[box]]
        inputs = self.proc(images=image, return_tensors="pt", **kw)
        if self._cache is None or self._cache[0] is not image:
            self._cache = (image, self.model.get_image_embeddings(inputs["pixel_values"].to(self.device)))
        out = self.model(
            image_embeddings=self._cache[1],
            input_points=inputs["input_points"].to(self.device) if points else None,
            input_labels=inputs["input_labels"].to(self.device) if points else None,
            input_boxes=inputs["input_boxes"].to(self.device) if box is not None else None,
            multimask_output=False,
        )
        masks = out.pred_masks.cpu()
        logits = self.proc.post_process_masks(masks, inputs["original_sizes"], binarize=False)[0]
        return torch.sigmoid(logits[0, 0]).numpy()

In [ ]:
dino = Dino()
sam = Sam()
RESULTS["models"] = {"dino": DINO_SOURCE, "sam": SAM_ID, "device": DEVICE}

<table width="100%" bgcolor="#FFF8E1"><tr><td><font color="#5D4037">🧾 <b>DINOv3 licence notice.</b>
DINOv3 is licensed under the <a href="https://huggingface.co/timm/vit_small_patch16_dinov3.lvd1689m">DINOv3
License</a>, Copyright © Meta Platforms, Inc. All Rights Reserved. It allows research and commercial use
under Meta's conditions and acceptable-use policy; keep this notice if you redistribute the weights or
anything derived from them. The <code>timm</code> re-host used here is open to everyone, no account needed.
SAM 2.1 is Apache-2.0. Both are re-checked about four weeks before each event, since Meta can change the
terms.</font></td></tr></table>

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: the first slow step.</b>
The first time a model sees a new image size, PyTorch compiles kernels for it: the first call of the next
section is slower than the ones after. On the CPU the first SAM call on a 1024 px image takes ~20 s.
</font></td></tr></table>

## 1. Out of the box: pets

**Objective:** see a foundation model segment pets almost perfectly, zero-shot.

Three tools, no training data of ours: **SAM** segments whatever you point at, **DINOv3** turns an image
into a grid of features that already "know" what belongs together, and a **linear head** on those
features learns a new class from a handful of masks in seconds. Here they meet the easy task. Keep the
numbers you get: section 2 runs exactly the same three cells on the specialised task.

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: two kinds of
foundation model.</b> <b>SAM</b> (Segment Anything) is <i>promptable</i>: you give it an image and a
hint (a click, a box), and it returns the mask of the object the hint is on. Nothing is trained. It
learnt "what an object is" from a billion masks of everyday photos. <b>DINOv3</b> is a <i>feature
extractor</i>: it cuts the image into 16 × 16 px patches and returns a 384-number description of each
patch, learnt without any labels. Two patches with similar descriptions look alike to the model. On
their own the features segment nothing; a tiny classifier on top of them does, and that is the
<b>linear head</b> we train below.</font></td></tr></table>

This helper cell defines the two pieces this section measures with: how to turn a person's hint into a
SAM prompt, and `show_masks()`, the figure used for every comparison in the notebook (prediction in
violet, ground truth in green, prompts in teal).

In [ ]:
# @title Prompts and figure helpers { display-mode: "form" }
from safetensors import safe_open
from safetensors.torch import load_file

_FEATURE_FILES: dict = {}


def click_point(item: dict) -> list[float]:
    """One (x, y) where a person would click: a pixel of the first foreground scribble stroke."""
    ys, xs = np.nonzero(item["scribbles"] == 1)
    rng = np.random.default_rng([SEED, len(ys)])
    k = int(rng.integers(len(ys)))
    return [float(xs[k]), float(ys[k])]


def box_prompt(item: dict) -> list[float]:
    """The box a person would draw around the object: the full mask's bounding box, as x0, y0, x1, y1."""
    ys, xs = np.nonzero(item["mask"])
    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]


def show_masks(panels: list[tuple[str, np.ndarray]], prompts: dict | None = None, width: float = 3.2):
    """One row of RGB panels (title, image). `prompts` = {"point": [x, y], "box": [x0, y0, x1, y1]}."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width))
    axes = np.atleast_1d(axes)
    for ax, (title, img) in zip(axes, panels, strict=True):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    if prompts:
        for ax in axes:
            if prompts.get("point") is not None:
                x, y = prompts["point"]
                ax.plot(x, y, marker="*", markersize=14, color=THEME["scribble"], markeredgecolor="white")
            if prompts.get("box") is not None:
                x0, y0, x1, y1 = prompts["box"]
                rect = mpl.patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=2)
                rect.set_edgecolor(THEME["scribble"])
                ax.add_patch(rect)
    plt.tight_layout()
    plt.show()


def features_for(dataset: dict, item: dict, long_side: int) -> torch.Tensor:
    """DINOv3 patch features [384, h, w]: from the precomputed file when it exists, else computed now."""
    prefix = "pets" if dataset is PETS else "specialised"
    path = PRECOMPUTED / "features" / f"{prefix}_{long_side}.safetensors"
    if path.is_file():
        if path not in _FEATURE_FILES:
            _FEATURE_FILES[path] = safe_open(path, framework="pt")
        store = _FEATURE_FILES[path]
        if item["name"] in store.keys():
            return store.get_tensor(item["name"]).to(DEVICE).float()
    return dino.features(item["image"], long_side)


def pca_rgb(feat: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Project the 384-dim patch features onto their 3 main directions and show them as red, green, blue."""
    d, h, w = feat.shape
    x = feat.reshape(d, h * w).T.cpu().numpy().astype(np.float64)
    x = x - x.mean(0)
    _, _, vt = np.linalg.svd(x, full_matrices=False)
    rgb = x @ vt[:3].T
    rgb = (rgb - rgb.min(0)) / (rgb.max(0) - rgb.min(0) + 1e-9)
    img = torch.from_numpy(rgb.reshape(h, w, 3)).permute(2, 0, 1)[None].float()
    img = F.interpolate(img, size=shape, mode="bilinear", align_corners=False)[0].permute(1, 2, 0)
    return (img.numpy() * 255).astype(np.uint8)

This cell loads the held-out pets, the evaluation set every approach in this notebook is scored on, and
fixes the resolution DINOv3 works at. On a CPU, 12 of them keep each cell under a minute.

In [ ]:
PETS["feat_side"] = 256 if SMOKE else PETS["index"]["feature_long_side"]
PETS["eval"] = PETS["held"] if (SMOKE or DEVICE == "cuda") else PETS["held"][:12]
PETS_EVAL = [load_item(PETS, it) for it in PETS["eval"]]
print(f"evaluation set: {len(PETS_EVAL)} held-out pets · DINOv3 features at {PETS['feat_side']} px long side")

### 1.1 SAM: one click, one box

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The next
cell gives SAM a single click on a pet, then a box around it. Before you run it: will one click be
enough for a clean mask of the whole animal, and what could go wrong (fur vs floor, the tail, a second
animal)? Say it to your neighbour or write it down.</font></td></tr></table>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next
cell and look at the three panels.</font></td></tr></table>

**Q2.** 🟡 Where do the one-click mask and the box mask differ from the ground truth, if at all? Is the
difference at the object's boundary, or is a whole part missing or added? Try the same cell on another
pet by changing the index.

```python
demo = PETS_EVAL[0]   # try 1, 2, ...
```

This cell runs SAM twice on one held-out pet, once with a single click and once with a box, and shows
both masks next to the ground truth, so you can see what a promptable model gives for free.

In [ ]:
# Q2 — SAM with one click and with one box on a held-out pet
demo = PETS_EVAL[0]  # try 1, 2, ...
point, box = click_point(demo), box_prompt(demo)
t0 = time.monotonic()
sam_point = sam.probability(demo["image"], points=[point], labels=[1]) >= 0.5
sam_box = sam.probability(demo["image"], box=box) >= 0.5
print(f"{demo['name']}: two SAM calls in {time.monotonic() - t0:.1f} s on {DEVICE}")
show_masks(
    [
        ("SAM, one click", overlay(demo["image"], sam_point, THEME["works"])),
        ("SAM, one box", overlay(demo["image"], sam_box, THEME["works"])),
        ("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])),
    ],
    prompts={"point": point, "box": box},
)

### 1.2 One number: IoU

Looking at masks is the honest way to judge a segmentation, and a single number lets us compare
approaches on a hundred images. We use one metric for the whole workshop: **IoU** (intersection over
union), the overlap between the prediction and the ground truth divided by the area covered by either.
1.0 is a perfect match, 0.0 means no overlap. Section 2 adds one extra metric for thin structures.

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your turn (optional, 2 min).</b>
Fill <code>my_iou</code>: two boolean arrays in, one float out. Pixels where both are True are the
intersection; pixels where at least one is True are the union. If the union is empty, return 1.0 (two
empty masks agree). The cell below checks your version on a 4 × 4 play example and continues with
ours if you skip it.</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def my_iou(pred: np.ndarray, truth: np.ndarray) -> float:
    """Intersection over union of two boolean masks of the same shape; 1.0 when both are empty."""
    ...


def _ref_iou(pred: np.ndarray, truth: np.ndarray) -> float:
    """Reference: |pred ∧ truth| / |pred ∨ truth|."""
    inter, union = float((pred & truth).sum()), float((pred | truth).sum())
    return inter / union if union else 1.0


_play_pred = np.zeros((4, 4), bool)
_play_pred[1:3, 1:4] = True  # 6 pixels
_play_truth = np.zeros((4, 4), bool)
_play_truth[0:3, 1:3] = True  # 6 pixels, 4 shared -> IoU = 4 / 8
iou = use(my_iou, reference=_ref_iou, check=lambda f: round(float(f(_play_pred, _play_truth)), 4), name="iou")

This cell scores SAM's one-click and one-box masks on every evaluation pet with the IoU you just
defined, and keeps the mean as this task's zero-shot number: section 2 compares it with the
specialised task.

In [ ]:
t0 = time.monotonic()
zero_shot = {"point": [], "box": []}
for it in PETS_EVAL:
    pt = click_point(it)
    zero_shot["point"].append(iou(sam.probability(it["image"], points=[pt], labels=[1]) >= 0.5, it["mask"]))
    zero_shot["box"].append(iou(sam.probability(it["image"], box=box_prompt(it)) >= 0.5, it["mask"]))
zero_shot = {k: float(np.mean(v)) for k, v in zero_shot.items()}
RESULTS["pets_zero_shot_iou"] = zero_shot["box"]  # the headline: SAM's best single prompt, one box
RESULTS["pets_zero_shot_point_iou"] = zero_shot["point"]
RESULTS["pets_zero_shot_box_iou"] = zero_shot["box"]
RESULTS["pets_zero_shot_seconds"] = round(time.monotonic() - t0, 1)
print(
    f"SAM zero-shot on {len(PETS_EVAL)} held-out pets · mean IoU one click {zero_shot['point']:.3f} · "
    f"one box {zero_shot['box']:.3f} · {RESULTS['pets_zero_shot_seconds']} s"
)

### 1.3 What DINOv3 sees

This cell takes the 384-number description DINOv3 gives each 16 × 16 px patch, keeps the three
directions that vary most (a PCA) and shows them as red, green and blue, so you can see what the model
groups together before any training.

In [ ]:
fig_items = [load_item(PETS, it) for it in PETS["fit"][:3]]
panels = []
for it in fig_items:
    feat = features_for(PETS, it, PETS["feat_side"])
    title = f"DINOv3 PCA · {feat.shape[2]}×{feat.shape[1]} patches"
    panels += [(it["name"], it["image"]), (title, pca_rgb(feat, it["image"].shape[:2]))]
show_masks(panels, width=2.6)

**Q3.** 🟡 In the PCA images, which pixels share a colour: the whole animal, parts of it (head, fur,
legs), or the floor and the background too? Does the colour follow the object's outline, and how fine is
the grid behind it?

<details><summary>What to look for</summary>

The colours are not classes: they are the three directions along which the features vary most across
this image, so the same colour means "the model describes these patches alike". Typically the animal
separates from the background in one direction and its parts (face vs body) in another, and the blocky
edges show the 16 px grid: the features know <i>what</i> is where, at patch resolution, not <i>exactly
where</i> the boundary is. Whether that is enough depends on the size of what you want to segment.

</details>

### 1.4 A linear head on five pets

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The next
cells train a classifier with 385 parameters on the patch features of just <b>five</b> pets with full
masks, then score it on the held-out ones. Will its IoU land above, near or below SAM's one-click number
from 1.2? Write down a number before you run.</font></td></tr></table>

This helper cell defines the head used for every training step in the notebook: a 1 × 1 convolution on
standardised patch features, trained with a balanced cross-entropy that ignores pixels labelled −1, plus
`load_head()` for our precomputed checkpoints.

In [ ]:
# @title Linear head: train, predict, load a checkpoint { display-mode: "form" }
def train_head(
    feats: list[torch.Tensor], labels: list[np.ndarray], steps: int, lr: float = 0.05, balanced: bool = True
) -> tuple[dict, float]:
    """Linear head (384 -> 1) on standardised patch features. Labels per pixel: 1 fg, 0 bg, -1 ignored.

    Returns the head's tensors (weight, bias, feat_mean, feat_std) and the training seconds.
    """
    torch.manual_seed(SEED)
    allf = torch.cat([f.flatten(1) for f in feats], 1)
    mu, sd = allf.mean(1), allf.std(1) + 1e-6
    feats = [(f - mu.view(-1, 1, 1)) / sd.view(-1, 1, 1) for f in feats]
    labs = [torch.from_numpy(np.asarray(lab, dtype=np.int8)).to(DEVICE) for lab in labels]
    n_pos = sum(int((lab == 1).sum()) for lab in labs)
    n_neg = sum(int((lab == 0).sum()) for lab in labs)
    if balanced:
        w_pos, w_neg = 0.5 / max(n_pos, 1), 0.5 / max(n_neg, 1)
    else:
        w_pos = w_neg = 1.0 / max(n_pos + n_neg, 1)
    head = torch.nn.Conv2d(feats[0].shape[0], 1, 1).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    t0 = time.monotonic()
    for _ in range(steps):
        opt.zero_grad()
        loss = 0.0
        for f, lab in zip(feats, labs, strict=True):
            logit = F.interpolate(head(f[None]), size=lab.shape, mode="bilinear", align_corners=False)[0, 0]
            w = torch.where(lab == 1, w_pos, torch.where(lab == 0, w_neg, 0.0))
            bce = F.binary_cross_entropy_with_logits(logit, (lab == 1).float(), reduction="none")
            loss = loss + (bce * w).sum()
        loss.backward()
        opt.step()
    tensors = {
        "weight": head.weight.detach().cpu(),
        "bias": head.bias.detach().cpu(),
        "feat_mean": mu.cpu(),
        "feat_std": sd.cpu(),
    }
    return tensors, time.monotonic() - t0


@torch.no_grad()
def predict_head(tensors: dict, feat: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Apply a head to one feature grid; returns the boolean foreground mask at `shape` (logit > 0)."""
    dev = feat.device
    f = (feat - tensors["feat_mean"].to(dev).view(-1, 1, 1)) / tensors["feat_std"].to(dev).view(-1, 1, 1)
    logit = F.conv2d(f[None], tensors["weight"].to(dev), tensors["bias"].to(dev))
    logit = F.interpolate(logit, size=shape, mode="bilinear", align_corners=False)[0, 0]
    return (logit > 0).cpu().numpy()


def load_head(dataset: dict, regime: str, long_side: int | None = None) -> tuple[dict, dict]:
    """Our precomputed checkpoint for (dataset, regime): its tensors and its index entry."""
    long_side = long_side or dataset["feat_side"]
    key = "pets" if dataset is PETS else SPECIALISED["key"]
    wanted = (key, regime, long_side)
    heads = PRECOMPUTED_HEADS
    entry = next((h for h in heads if (h["dataset"], h["regime"], h["long_side"]) == wanted), None)
    if entry is None:
        raise SystemExit(f"no precomputed head for {wanted}. Run setup_check.ipynb first.")
    return load_file(PRECOMPUTED / entry["file"]), entry

This cell trains the head on the full masks of five fit pets (sections 1 and 2 are the only places
training sees full masks) and prints the seconds it took. Not done in a minute, or on a CPU and in a
hurry? Set `TRAIN_YOURSELF = False` and run it again: it loads our checkpoint instead.

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed head "full5" instead of training
N_TRAIN_PETS, HEAD_STEPS = 5, 30 if SMOKE else 200
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

train_items = [load_item(PETS, it) for it in PETS["fit"][:N_TRAIN_PETS]]
train_feats = [features_for(PETS, it, PETS["feat_side"]) for it in train_items]
if TRAIN_YOURSELF:
    train_labels = [it["mask"].astype(np.int8) for it in train_items]  # full masks: allowed in section 1 only
    head_full5, head_sec = train_head(train_feats, train_labels, HEAD_STEPS)
    head_src = f"trained here on {len(train_items)} pets, {HEAD_STEPS} steps"
else:
    head_full5, entry = load_head(PETS, "full5")
    head_sec = entry["train_sec"]
    head_src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} pets)"
print(f"linear head · {head_src} · {head_sec:.1f} s on {DEVICE}")
if head_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

This cell scores the head on the same evaluation pets as SAM, keeps the mean IoU, and shows three held-out
pets with the head's mask next to the ground truth.

In [ ]:
t0 = time.monotonic()
head_preds = [
    predict_head(head_full5, features_for(PETS, it, PETS["feat_side"]), it["mask"].shape) for it in PETS_EVAL
]
head_ious = [iou(pred, it["mask"]) for pred, it in zip(head_preds, PETS_EVAL, strict=True)]
RESULTS["pets_head_iou"] = float(np.mean(head_ious))
RESULTS["pets_head"] = {
    "iou": RESULTS["pets_head_iou"],
    "n_train": len(train_items),
    "train_sec": round(float(head_sec), 2),
    "infer_sec": round(time.monotonic() - t0, 1),
}
print(
    f"linear head on {len(PETS_EVAL)} held-out pets · mean IoU {RESULTS['pets_head_iou']:.3f} "
    f"(SAM one click: {RESULTS['pets_zero_shot_point_iou']:.3f}, "
    f"one box: {RESULTS['pets_zero_shot_box_iou']:.3f})"
)
panels = []
for it, pred, v in list(zip(PETS_EVAL, head_preds, head_ious, strict=True))[:3]:
    panels += [
        (f"head · IoU {v:.2f}", overlay(it["image"], pred, THEME["works"])),
        ("ground truth", overlay(it["image"], it["mask"], THEME["truth"])),
    ]
show_masks(panels, width=2.6)

<table width="100%" bgcolor="#E8EAF6"><tr><td><font color="#1A237E">🔀 <b>Across approaches.</b> Two ways
to the same mask, scored on the same hidden ground truth: SAM needs <i>one hint per image</i> and no
training; the head needs <i>five full masks once</i> and then runs on every image with no hint. Keep both
numbers: the comparison table in section 8 starts with them, and section 2 reruns both on the
specialised task.</font></td></tr></table>

**Q4.** 🟡 Compare the two numbers you got with the prediction you wrote down. Which approach would you
pick for 10 000 pet photos, and which for 10 photos? What does each one ask of a person?

<details><summary>One possible answer</summary>

For 10 photos, a click each is nothing, and SAM's masks come with sharp boundaries. For 10 000, a person
cannot click on each one, while the head, once trained on a few masks, runs with no hint at all. Its
boundaries follow the 16 px patch grid, so it loses a little IoU at the edges, which matters more as the
objects get smaller. Hardware or seed may change the numbers; the trade-off stays.

</details>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">👥 <b>Optional, 3 min.</b> Compare
the demo pet with your neighbour's: did one click fail on either of yours? If it did, was the click on
an ambiguous spot (fur against a similar floor, a collar, a second animal)?</font></td></tr></table>

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: a clean mask is not
a correct one.</b> SAM always returns a confident-looking mask, whatever you click on. On pets that is
usually the right object. Keep that habit of looking at the mask next to the ground truth: in the next
section it is the only thing that shows the model has quietly segmented the wrong thing.
</font></td></tr></table>

## 2. Where it falls short

**Objective:** run the same tools on the specialised task and probe SAM with escalating prompts.

Same three tools, same cells, new task. Section 1 gave you three numbers on pets: SAM with one click,
SAM with one box, and a linear head on five full masks. Here they meet the specialised images, where the
objects are thin, branching and low-contrast. Then we push SAM with more and more prompts, and give
DINOv3 twice the resolution, to see which of the two limits the result.

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: why thin structures
are hard.</b> DINOv3 describes the image in 16 × 16 px patches. A vessel 5 px wide covers a third of a
patch, so its description is mostly "background with a dark line in it", and the head can only say
"vessel" or "not" for the whole square. SAM sees finer detail, but it was trained on a billion masks of
<i>objects</i>: compact things with a clear inside and outside. A vessel tree has
no inside, and one click on it is ambiguous (this vessel? the whole tree? the eye?).
</font></td></tr></table>

This helper cell defines what this section measures with: the evaluation region (the fundus, not the
black border around it), skeletons for the thin-structure metric, clicks spread along a person's
scribbles, and a figure that marks every click.

In [ ]:
# @title Thin-structure helpers: region, skeleton, clicks, figure { display-mode: "form" }
from skimage.morphology import skeletonize


def region(item: dict) -> np.ndarray:
    """Where we score: the field of view if the dataset has one, else the whole image."""
    return item["fov"] if item.get("fov") is not None else np.ones_like(item["mask"])


def skeleton(mask: np.ndarray) -> np.ndarray:
    """The one-pixel-wide centre line of a boolean mask."""
    return skeletonize(np.asarray(mask, bool))


def scribble_points(item: dict, n: int) -> list[list[float]]:
    """n (x, y) clicks along the foreground scribbles: the section-1 click first, then spread out as far as
    possible from each other (farthest-point sampling), the way a person would click along the vessels."""
    ys, xs = np.nonzero(item["scribbles"] == 1)
    pts = np.stack([xs, ys], 1).astype(float)
    chosen = [np.array(click_point(item))]
    dist = np.linalg.norm(pts - chosen[0], axis=1)
    while len(chosen) < min(n, len(pts)):
        k = int(dist.argmax())
        chosen.append(pts[k])
        dist = np.minimum(dist, np.linalg.norm(pts - pts[k], axis=1))
    return [[float(x), float(y)] for x, y in chosen]


def score(pred: np.ndarray, item: dict) -> dict:
    """IoU and clDice of a boolean prediction, both inside the evaluation region."""
    r = region(item)
    return {"iou": iou(pred & r, item["mask"] & r), "cldice": cldice(pred & r, item["mask"] & r)}


def mean_scores(scores: list[dict]) -> dict:
    return {k: float(np.mean([s[k] for s in scores])) for k in scores[0]}


def show_clicks(panels: list[tuple[str, np.ndarray, list | None]], width: float = 2.6):
    """One row of panels (title, RGB image, clicks or None); clicks are drawn as teal stars."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width))
    for ax, (title, img, pts) in zip(np.atleast_1d(axes), panels, strict=True):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
        for x, y in pts or []:
            ax.plot(x, y, marker="*", markersize=10, color=THEME["scribble"], markeredgecolor="white")
    plt.tight_layout()
    plt.show()

This cell loads the held-out specialised images, the evaluation set for the rest of the notebook, and
prints how much of the fundus the structures cover: keep that number, it comes back as a baseline.

In [ ]:
SPECIALISED["feat_side"] = 256 if SMOKE else SPECIALISED["feature_long_side"]
SPECIALISED["eval"] = SPECIALISED["held"] if (SMOKE or DEVICE == "cuda") else SPECIALISED["held"][:12]
SPEC_EVAL = [load_item(SPECIALISED, it) for it in SPECIALISED["eval"]]
fg_share = float(np.mean([it["mask"][region(it)].mean() for it in SPEC_EVAL]))
print(
    f"evaluation set: {len(SPEC_EVAL)} held-out images · {SPECIALISED['task']} · the foreground covers "
    f"{fg_share:.0%} of the evaluation region · DINOv3 features at {SPECIALISED['feat_side']} px long side"
)

### 2.1 SAM: one click, one box, again

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The next
cell is the section-1 cell on a specialised image: one click on a vessel (a pixel of a person's
scribble), then a box around all of them. Before you run it: what will SAM segment from one click on a
thin line? And what is "the object" inside a box that covers the whole image?</font></td></tr></table>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next
cell and compare the panels with the pets in section 1.</font></td></tr></table>

**Q5.** 🟡 What did SAM segment from the click, and from the box: a vessel, part of the tree, a blob, the
whole fundus? Try two other images by changing the index (on a CPU session there are 12).

```python
demo = SPEC_EVAL[0]   # try 1, 2, ...
```

This cell runs SAM twice on one held-out specialised image, once with the click and once with the box,
and shows both masks in orange next to the ground truth.

In [ ]:
# Q5 — SAM with one click and with one box on a held-out specialised image
demo = SPEC_EVAL[0]  # try 1, 2, ...
point, box = click_point(demo), box_prompt(demo)
t0 = time.monotonic()
sam_point = sam.probability(demo["image"], points=[point], labels=[1]) >= 0.5
sam_box = sam.probability(demo["image"], box=box) >= 0.5
print(f"{demo['name']}: two SAM calls in {time.monotonic() - t0:.1f} s on {DEVICE}")
show_masks(
    [
        ("SAM, one click", overlay(demo["image"], sam_point, THEME["fails"])),
        ("SAM, one box", overlay(demo["image"], sam_box, THEME["fails"])),
        ("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])),
    ],
    prompts={"point": point, "box": box},
)

### 2.2 A second number for thin structures: clDice

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: clDice.</b> IoU counts
pixels, and on thin structures pixels mislead. Predict every vessel twice as wide as it is and IoU drops
to about 0.5, although nothing was missed; miss all the capillaries and IoU barely moves, because they
hold few pixels. <b>clDice</b> (Shit et al., CVPR 2021) compares <i>centre lines</i> instead. It takes
the skeleton of each mask and asks two questions: which share of the predicted centre line lies inside
the true vessels (<i>topology precision</i>), and which share of the true centre line the prediction
covers (<i>topology sensitivity</i>). clDice is their harmonic mean: 1.0 when the tree is traced
exactly, low when branches are missing or invented. We report it next to IoU for the specialised task.
</font></td></tr></table>

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your turn (optional, 3 min).</b>
Fill <code>my_cldice</code>: two boolean masks in, one float out. <code>skeleton(mask)</code> gives the
centre line. Topology precision = pixels of <code>skeleton(pred)</code> inside <code>truth</code>, divided
by the pixels of <code>skeleton(pred)</code>; topology sensitivity is the same with the roles swapped.
Return 0.0 when either skeleton is empty. The cell checks your version on two play masks, a good trace
and a blob that covers everything, and continues with ours if you skip it.</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def my_cldice(pred: np.ndarray, truth: np.ndarray) -> float:
    """clDice of two boolean masks: 2·tprec·tsens / (tprec + tsens); 0.0 when either skeleton is empty."""
    ...


def _ref_cldice(pred: np.ndarray, truth: np.ndarray) -> float:
    """Reference: harmonic mean of topology precision and topology sensitivity."""
    sk_pred, sk_truth = skeleton(pred), skeleton(truth)
    if not sk_pred.any() or not sk_truth.any():
        return 0.0
    tprec = float((sk_pred & truth).sum() / sk_pred.sum())
    tsens = float((sk_truth & pred).sum() / sk_truth.sum())
    return 2 * tprec * tsens / (tprec + tsens) if tprec + tsens else 0.0


_play_truth = np.zeros((12, 12), bool)
_play_truth[2:4, :] = True  # a horizontal vessel, 2 px wide
_play_truth[:, 9:11] = True  # crossed by a vertical one
_play_trace = np.zeros((12, 12), bool)
_play_trace[1:5, :] = True  # the horizontal vessel, drawn too wide; the vertical one missed
_play_blob = np.ones((12, 12), bool)  # "everything is vessel"
cldice = use(
    my_cldice,
    reference=_ref_cldice,
    check=lambda f: [round(float(f(p, _play_truth)), 4) for p in (_play_trace, _play_blob)],
    name="cldice",
)
print(
    f"play example · IoU / clDice of the wide trace: {iou(_play_trace, _play_truth):.2f} / "
    f"{cldice(_play_trace, _play_truth):.2f} · of the blob: {iou(_play_blob, _play_truth):.2f} / "
    f"{cldice(_play_blob, _play_truth):.2f}"
)

### 2.3 Escalating prompts: 1 → 5 → 20 clicks, and a box

If one click is ambiguous, maybe more clicks tell SAM what we mean. The next cell gives it 1, 5 and 20
clicks spread along the scribbles a person drew, then the box, on every evaluation image. It also scores
the laziest possible answer, a mask that says "everything is foreground": a prediction no better than
that line has learnt nothing about the task.

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> Write down
what you expect from 1 to 20 clicks: will IoU rise steadily, jump at some point, or stay flat? Will the
box beat 20 clicks? And where will the grey "everything" line sit?</font></td></tr></table>

This cell runs the prompt ladder on every evaluation image (one SAM image encoding per image, then four
cheap prompt decodings) and keeps the mean IoU, clDice and the share of the fundus SAM marks per setting.

In [ ]:
LADDER = {"1 point": 1, "5 points": 5, "20 points": 20, "box": None}  # setting -> number of clicks
t0 = time.monotonic()
ladder_scores = {name: [] for name in LADDER}
ladder_cover = {name: [] for name in LADDER}
ladder_demo = {}  # the demo image's masks and clicks, for the figure below
sam_box_masks = []  # every image's box mask, for the gallery in 2.6
everything = []
for it in SPEC_EVAL:
    r = region(it)
    for name, n in LADDER.items():
        pts = scribble_points(it, n) if n else None
        if n:
            pred = sam.probability(it["image"], points=pts, labels=[1] * len(pts)) >= 0.5
        else:
            pred = sam.probability(it["image"], box=box_prompt(it)) >= 0.5
        ladder_scores[name].append(score(pred, it))
        ladder_cover[name].append(float(pred[r].mean()))
        if it is demo:
            ladder_demo[name] = (pred, pts)
    sam_box_masks.append(pred)  # the last setting is the box
    everything.append(score(r.copy(), it))
ladder_mean = {name: mean_scores(s) for name, s in ladder_scores.items()}
everything_mean = mean_scores(everything)
spec_sec = round(time.monotonic() - t0, 1)

RESULTS["sam_prompt_ladder"] = {name: v["iou"] for name, v in ladder_mean.items()}
RESULTS["sam_prompt_ladder_cldice"] = {name: v["cldice"] for name, v in ladder_mean.items()}
RESULTS["specialised_zero_shot_iou"] = ladder_mean["box"]["iou"]  # same headline as pets: one box
RESULTS["specialised_zero_shot_point_iou"] = ladder_mean["1 point"]["iou"]
RESULTS["specialised_zero_shot_box_iou"] = ladder_mean["box"]["iou"]
RESULTS["specialised_everything_iou"] = everything_mean["iou"]
RESULTS["specialised_zero_shot_seconds"] = spec_sec

print(f"SAM prompt ladder on {len(SPEC_EVAL)} held-out images · {spec_sec} s on {DEVICE}")
print(f"{'setting':<12}{'IoU':>7}{'clDice':>8}{'marked':>8}")
for name, v in ladder_mean.items():
    print(f"{name:<12}{v['iou']:>7.3f}{v['cldice']:>8.3f}{np.mean(ladder_cover[name]):>8.0%}")
print(f"{'everything':<12}{everything_mean['iou']:>7.3f}{everything_mean['cldice']:>8.3f}{1:>8.0%}")

fig, ax = plt.subplots(figsize=(6.4, 3))
names = list(ladder_mean)
ax.bar(names, [ladder_mean[k]["iou"] for k in names], color=THEME["fails"], label="specialised, SAM")
ax.axhline(everything_mean["iou"], color=THEME["background"], ls="--", label='"everything is foreground"')
ax.axhline(RESULTS["pets_zero_shot_iou"], color=THEME["works"], ls=":", label="pets, SAM one box (section 1)")
ax.set_ylabel("mean IoU")
ax.set_ylim(0, 1)
ax.set_title("More prompts, same model")
ax.legend(fontsize=8, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

show_clicks(
    [(f"SAM, {k}", overlay(demo["image"], m, THEME["fails"]), p) for k, (m, p) in ladder_demo.items()]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]), None)]
)

**Q6.** 🟡 Compare the ladder with your prediction. Did more clicks help, and did the masks get closer to
the vessels or just bigger (look at the "marked" column and the grey line)? What does that tell you about
what SAM thinks an object is?

<details><summary>What to look for</summary>

Read the IoU next to the grey line, not on its own. A mask that marks most of the fundus scores exactly
the "everything" baseline, whatever the prompt. If more clicks push the "marked" share up while IoU stays
near the line, SAM is answering "the object that contains all these points", and for a vessel tree that
object is the eye. If clicks do help on your images, look at where: usually along the thick vessels near
the clicks, rarely the capillaries in between. SAM has no notion of "all the thin dark lines"; a prompt
can only say <i>where</i>, not <i>what kind of thing</i>. Section 7 comes back to prompting with
background clicks too.

</details>

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: over-trusting SAM's
confident masks.</b> SAM returns a crisp mask for every prompt, and its own confidence score is high for
a mask of the whole eye, because the whole eye <i>is</i> a good object. Nothing in the output says "this
is the wrong kind of thing". Always score against a baseline that knows nothing (the grey line), and
look at the mask.</font></td></tr></table>

### 2.4 What DINOv3 sees here

This cell shows the same 3-colour PCA of the DINOv3 patch features as section 1.3, now on specialised
images, so you can see whether the features separate the vessels from the background before any training.

In [ ]:
fig_items = [load_item(SPECIALISED, it) for it in SPECIALISED["fit"][:3]]
panels = []
for it in fig_items:
    feat = features_for(SPECIALISED, it, SPECIALISED["feat_side"])
    title = f"DINOv3 PCA · {feat.shape[2]}×{feat.shape[1]} patches"
    panels += [(it["name"], it["image"]), (title, pca_rgb(feat, it["image"].shape[:2]))]
show_masks(panels, width=2.6)

**Q7.** 🟡 Do the PCA colours follow the vessels, or something else (the bright optic disc, the dark
border, the overall brightness)? Compare with the pets, where one direction separated animal from floor.

<details><summary>What to look for</summary>

The three main directions of variation describe what varies most across the image, and on a fundus that
is rarely the vessels: it is the optic disc, the illumination falling off towards the edge, the border.
Thick vessels may show as a faint trace of blocks; thin ones vanish inside the 16 px squares. That does
not mean the features hold no vessel information (a trained head can still find a direction for it), but
it is no longer the first thing the model "sees".

</details>

### 2.5 The linear head, and the resolution it looks at

The head from section 1.4, trained the same way on the full masks of five specialised images. This is
the last time training sees a full mask: from section 3 on, only incomplete labels. We train it twice,
on DINOv3 features computed at two resolutions. At a 512 px long side, each 16 px patch covers 32 × 32
pixels of the 1024 px crop; at 1024 px, 16 × 16. (DINOv2 papers use 518 px: 37 patches of 14 px. DINOv3's
patches are 16 px, so 512 is the same idea.) Double the side and you get four times the patches, and
roughly four to sixteen times the compute.

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> With
<i>perfect</i> labels, will the head beat SAM here? And will doubling the resolution help IoU, clDice,
both or neither? Write two guesses.</font></td></tr></table>

This cell trains one head per resolution on the five fit images and prints the seconds it took. Not done
in a minute, or on a CPU and in a hurry? Set `TRAIN_YOURSELF = False` and run it again: it loads our
checkpoints (trained the same way on all fit images).

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "full" heads instead of training
N_TRAIN_SPEC, HEAD_STEPS = 5, 30 if SMOKE else 200
SIDES = SPECIALISED["feature_long_sides"]  # [512, 1024]: the two resolutions we compare
SIDES = [s // 4 for s in SIDES] if SMOKE else SIDES  # smoke thumbnails are 256 px
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

spec_train = [load_item(SPECIALISED, it) for it in SPECIALISED["fit"][:N_TRAIN_SPEC]]
spec_heads = {}  # long side -> (head tensors, training seconds)
for side in SIDES:
    if TRAIN_YOURSELF:
        feats = [features_for(SPECIALISED, it, side) for it in spec_train]
        labels = [np.where(region(it), it["mask"], -1).astype(np.int8) for it in spec_train]  # border ignored
        spec_heads[side] = train_head(feats, labels, HEAD_STEPS)
        src = f"trained here on {len(spec_train)} images, {HEAD_STEPS} steps"
    else:
        tensors, entry = load_head(SPECIALISED, "full", side)
        spec_heads[side] = (tensors, entry["train_sec"])
        src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
    print(f"linear head @ {side} px · {src} · {spec_heads[side][1]:.1f} s on {DEVICE}")
if sum(sec for _, sec in spec_heads.values()) > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoints next time.")

This cell scores both heads on the evaluation images with IoU and clDice, and zooms into the centre of the
demo image so you can compare the two resolutions on the thinnest vessels.

In [ ]:
t0 = time.monotonic()
head_preds, res_scores = {}, {}
for side, (tensors, _) in spec_heads.items():
    head_preds[side] = [
        predict_head(tensors, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
    ]
    res_scores[side] = mean_scores([score(p, it) for p, it in zip(head_preds[side], SPEC_EVAL, strict=True)])
lo, hi = min(SIDES), max(SIDES)
RESULTS["specialised_head_iou"] = res_scores[hi]["iou"]
RESULTS["specialised_head"] = {
    "iou": res_scores[hi]["iou"],
    "cldice": res_scores[hi]["cldice"],
    "long_side": hi,
    "n_train": len(spec_train),
    "train_sec": round(float(spec_heads[hi][1]), 2),
    "infer_sec": round(time.monotonic() - t0, 1),
}
RESULTS["resolution_effect"] = {f"{side} px": v for side, v in res_scores.items()}
for side, v in res_scores.items():
    print(f"linear head @ {side:>4} px · mean IoU {v['iou']:.3f} · clDice {v['cldice']:.3f}")
print(f"SAM one box · mean IoU {ladder_mean['box']['iou']:.3f} · clDice {ladder_mean['box']['cldice']:.3f}")

k = next(i for i, it in enumerate(SPEC_EVAL) if it is demo)
h, w = demo["mask"].shape
zoom = (slice(h // 4, 3 * h // 4), slice(w // 4, 3 * w // 4))  # the central quarter
show_masks(
    [
        (
            f"head @ {s} px · IoU {score(head_preds[s][k], demo)['iou']:.2f}",
            overlay(demo["image"], head_preds[s][k], THEME["fails"])[zoom],
        )
        for s in SIDES
    ]
    +[("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])[zoom])],
    width=3.0,
)

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: resolution is not
free.</b> Self-attention compares every patch with every other, so doubling the side multiplies the
patches by four and the attention cost by up to sixteen. At 1024 px, one specialised image is 4096 patches.
On the full 3504 px fundus it would be 48 000: that is why we work on crops and why the features are
precomputed. When the structure is thinner than a patch, check the resolution before anything else.
</font></td></tr></table>

### 2.6 The failure gallery

Numbers average the failures away; the gallery shows them. The next cell sorts the evaluation images
from worst to best (by the head's IoU at the higher resolution) and shows three at a time: the image, SAM
with a box, the head, and the ground truth. The last row puts the same models on the whole fundus, shrunk
to 1024 px, where the vessels are only 1–3 px wide.

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next
cell, then page through the gallery.</font></td></tr></table>

**Q8.** 🟡 For the worst images, is the failure the same kind for SAM and for the head? Look for: missed
thin vessels, vessels drawn too wide, the optic disc or the border marked as vessel, whole regions marked.
Page with the start index:

```python
GALLERY_START = 0   # 0 = the three worst; try 3, 6, ...
```

This cell draws the failure gallery from the predictions you already have (new model calls only for the
last row) and prints each image's IoU and clDice in the titles.

In [ ]:
# Q8 — the failure gallery, worst images first
GALLERY_START = 0  # 0 = the three worst; try 3, 6, ...
if not isinstance(GALLERY_START, int) or not 0 <= GALLERY_START < len(SPEC_EVAL):
    raise SystemExit(f"GALLERY_START must be a whole number from 0 to {len(SPEC_EVAL) - 1}.")
gallery = [score(p, it) for p, it in zip(head_preds[hi], SPEC_EVAL, strict=True)]
order = np.argsort([s["iou"] for s in gallery])
for k in order[GALLERY_START : GALLERY_START + 3]:
    it, sam_s, head_s = SPEC_EVAL[k], ladder_scores["box"][k], gallery[k]
    show_masks(
        [
            (it["name"], it["image"]),
            (
                f"SAM box · IoU {sam_s['iou']:.2f} · clDice {sam_s['cldice']:.2f}",
                overlay(it["image"], sam_box_masks[k], THEME["fails"]),
            ),
            (
                f"head @ {hi} px · IoU {head_s['iou']:.2f} · clDice {head_s['cldice']:.2f}",
                overlay(it["image"], head_preds[hi][k], THEME["fails"]),
            ),
            ("ground truth", overlay(it["image"], it["mask"], THEME["truth"])),
        ],
        width=2.8,
    )

# the whole fundus of the demo image, shrunk so its long side is 1024 px (smoke: 256 px)
meta = next(m for m in SPECIALISED["eval"] if m["name"] == demo["name"])
root = SPECIALISED["dir"]
full = {
    "image": np.array(Image.open(root / meta["full"]["image"]).convert("RGB")),
    "mask": read_png(root / meta["full"]["mask"]) > 127,
    "fov": read_png(root / meta["full"]["fov"]) > 127,
}
full_sam = sam.probability(full["image"], box=box_prompt(full)) >= 0.5
full_feat = dino.features(full["image"], max(full["image"].shape[:2]))
full_head = predict_head(spec_heads[hi][0], full_feat, full["mask"].shape)
full_masks = {"SAM box": full_sam, f"head @ {hi} px": full_head}
full_s = {name: score(m, full) for name, m in full_masks.items()}
show_masks(
    [(f"whole fundus · {meta['name']}", full["image"])]
    + [
        (f"{name} · IoU {full_s[name]['iou']:.2f} · clDice {full_s[name]['cldice']:.2f}",
         overlay(full["image"], m, THEME["fails"]))
        for name, m in full_masks.items()
    ]
    + [("ground truth", overlay(full["image"], full["mask"], THEME["truth"]))],
    width=2.8,
)
RESULTS["specialised_full_view"] = full_s

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">👥 <b>Pairs, 5 min: whose image
fails worst?</b> Each of you picks a different <code>GALLERY_START</code> or <code>demo</code> index and
finds the worst mask you can. Show it to your neighbour and name the failure in three words ("misses thin
vessels", "marks the disc", "the whole eye"). Is it the same failure for SAM and for the head?
</font></td></tr></table>

<details><summary>What people usually find</summary>

The two tools fail differently. SAM tends to fail <i>globally</i>: it marks a whole region, or one thick
vessel, because it looks for an object. The head fails <i>locally</i>: it finds the thick vessels, draws
them as wide as a patch, and loses the capillaries, because it can only label 16 px squares. On the
whole fundus both get worse, since every vessel is now thinner than a patch.

</details>

### 2.7 Pets vs the specialised task

This cell puts the section-1 numbers next to this section's, the same three tools on both tasks, so the
gap is one picture.

In [ ]:
rows = {
    "SAM, one click": (RESULTS["pets_zero_shot_point_iou"], RESULTS["specialised_zero_shot_point_iou"]),
    "SAM, one box": (RESULTS["pets_zero_shot_box_iou"], RESULTS["specialised_zero_shot_box_iou"]),
    "linear head": (RESULTS["pets_head_iou"], RESULTS["specialised_head_iou"]),
}
print(f"{'tool':<16}{'pets IoU':>10}{'specialised IoU':>17}")
for name, (p, s) in rows.items():
    print(f"{name:<16}{p:>10.3f}{s:>17.3f}")
print(f"{'everything':<16}{'':>10}{RESULTS['specialised_everything_iou']:>17.3f}")

x = np.arange(len(rows))
fig, ax = plt.subplots(figsize=(6.4, 3))
ax.bar(x - 0.2, [v[0] for v in rows.values()], 0.4, color=THEME["works"], label=PETS["name"])
ax.bar(x + 0.2, [v[1] for v in rows.values()], 0.4, color=THEME["fails"], label=SPECIALISED["name"])
ax.axhline(RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--",
           label='specialised, "everything is foreground"')
ax.set_xticks(x, list(rows))
ax.set_ylabel("mean IoU on held-out images")
ax.set_ylim(0, 1)
ax.set_title("Same tools, two tasks")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

<table width="100%" bgcolor="#E8EAF6"><tr><td><font color="#1A237E">🔀 <b>Across approaches.</b> Two
different limits showed up. SAM's limit is its idea of an object: more prompts make the mask bigger, not
thinner. The head's limit is the patch: even with perfect labels it can only answer per 16 px square, and
resolution is one lever (look at how far it moved IoU and clDice, and whether they agreed). From here on
the question changes from "does it work?" to "what information can we add, cheaply, to make it work?".
Perfect masks cost 3–5 hours per fundus, so the next sections only use cheap, incomplete ones.
</font></td></tr></table>

**Q9.** 🟡 You now have a small checklist for a new problem. Which two or three signs from this section
would tell you, on your own data, that a foundation model is not good enough out of the box?

<details><summary>One possible answer</summary>

(1) Its score sits near a baseline that knows nothing ("everything is foreground"), so compare with one.
(2) More or better prompts change the <i>size</i> of the mask but not its <i>shape</i>. (3) The objects
are smaller than a feature patch, and a higher resolution moves the numbers: a sign the representation,
not the labels, is the bottleneck. (4) A pixel metric and a structure metric (IoU vs clDice) disagree.
And always: the mask looks crisp and confident, so look at it next to the truth.

</details>

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Try it on your image (optional,
2 min).</b> Got a photo of something thin or specialised (a leaf's veins, a crack, a cable, a
micrograph)? Upload it with the folder icon on the left, put its path in <code>MY_IMAGE</code> and one or
more <code>(x, y)</code> clicks on the thing in <code>MY_CLICKS</code>, then run the cell. Left empty, it
runs on the demo image. Section 9 does this properly, with scribbles and the whole ladder.
</font></td></tr></table>

This cell runs SAM on your image (or the demo image) with your clicks and shows the mask, the quickest way
to see whether your own problem looks more like the pets or like this section.

In [ ]:
MY_IMAGE = ""  # e.g. "/content/my_photo.jpg"; empty = the demo image
MY_CLICKS = []  # e.g. [(120, 80), (300, 210)]; empty = the image centre
if not isinstance(MY_IMAGE, str) or not isinstance(MY_CLICKS, list):
    raise SystemExit('MY_IMAGE must be a path in quotes (or "") and MY_CLICKS a list of (x, y) pairs.')
if MY_IMAGE and not Path(MY_IMAGE).is_file():
    raise SystemExit(f"{MY_IMAGE} not found. Upload it with the folder icon on the left and copy its path.")
my_img = np.array(Image.open(MY_IMAGE).convert("RGB")) if MY_IMAGE else demo["image"]
my_h, my_w = my_img.shape[:2]
my_clicks = [[float(x), float(y)] for x, y in MY_CLICKS] or [[my_w / 2, my_h / 2]]
if not all(0 <= x < my_w and 0 <= y < my_h for x, y in my_clicks):
    raise SystemExit(f"every click must lie inside the image: 0 ≤ x < {my_w}, 0 ≤ y < {my_h}.")
my_mask = sam.probability(my_img, points=my_clicks, labels=[1] * len(my_clicks)) >= 0.5
show_clicks(
    [
        (MY_IMAGE or f"demo · {demo['name']}", my_img, my_clicks),
        (
            f"SAM, {len(my_clicks)} click(s) · marks {my_mask.mean():.0%}",
            overlay(my_img, my_mask, THEME["fails"]),
            my_clicks,
        ),
    ],
    width=3.2,
)